In [73]:
# Import original dataset

import pandas as pd

df = pd.read_csv("Invistico_Airline.csv")

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129880 entries, 0 to 129879
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   satisfaction                       129880 non-null  object 
 1   Gender                             129880 non-null  object 
 2   Customer Type                      129880 non-null  object 
 3   Age                                129880 non-null  int64  
 4   Type of Travel                     129880 non-null  object 
 5   Class                              129880 non-null  object 
 6   Flight Distance                    129880 non-null  int64  
 7   Seat comfort                       129880 non-null  int64  
 8   Departure/Arrival time convenient  129880 non-null  int64  
 9   Food and drink                     129880 non-null  int64  
 10  Gate location                      129880 non-null  int64  
 11  Inflight wifi service              1298

In [ ]:
print("test pull requested")

In [74]:
df.head()

,satisfaction,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Female,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Male,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Female,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Female,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Female,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,...,4,2,2,0,2,4,2,5,0,0.0


In [75]:
# Step 1: Validate and inspect data types

print("Data Types:")
print(df.dtypes)

Data Types:
satisfaction                          object
Gender                                object
Customer Type                         object
Age                                    int64
Type of Travel                        object
Class                                 object
Flight Distance                        int64
Seat comfort                           int64
Departure/Arrival time convenient      int64
Food and drink                         int64
Gate location                          int64
Inflight wifi service                  int64
Inflight entertainment                 int64
Online support                         int64
Ease of Online booking                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Online boarding                        int64
Departure Delay in Minutes             int6

In [76]:
# Step 2: Find missing values

print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
satisfaction                           0
Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Seat comfort                           0
Departure/Arrival time convenient      0
Food and drink                         0
Gate location                          0
Inflight wifi service                  0
Inflight entertainment                 0
Online support                         0
Ease of Online booking                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Online boarding                        0
Departure Delay in Minutes             0
Arrival Delay in Minutes             393
dtype: int64


In [77]:
# Step 3: Standardize text data (all object/categorical columns)

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.lower().str.strip()

print("Step 3: After text standardization:")
print(df.head())

Step 3: After text standardization:
  satisfaction  Gender   Customer Type  Age   Type of Travel     Class  \
0    satisfied  female  loyal customer   65  personal travel       eco   
1    satisfied    male  loyal customer   47  personal travel  business   
2    satisfied  female  loyal customer   15  personal travel       eco   
3    satisfied  female  loyal customer   60  personal travel       eco   
4    satisfied  female  loyal customer   70  personal travel       eco   

   Flight Distance  Seat comfort  Departure/Arrival time convenient  \
0              265             0                                  0   
1             2464             0                                  0   
2             2138             0                                  0   
3              623             0                                  0   
4              354             0                                  0   

   Food and drink  ...  Online support  Ease of Online booking  \
0               0  ...    

In [78]:
# Step 4: Binary encode 'satisfaction' column (creates a new column)

if 'satisfaction' in df.columns:
    df['satisfaction'] = df['satisfaction'].map({'satisfied': 1, 'dissatisfied': 0})

print("Step 4: After binary encoding 'satisfaction':")
print(df['satisfaction'].head())

Step 4: After binary encoding 'satisfaction':
0    1
1    1
2    1
3    1
4    1
Name: satisfaction, dtype: int64


In [79]:
# Step 5: One-hot encode all remaining categorical (object) columns

# Columns you want to keep as original (exclude from one-hot encoding)
exclude_categorical = ['Gender', 'Type of Travel']

# Identify which object columns to encode
cat_cols_to_encode = [col for col in df.select_dtypes(include='object').columns if col not in exclude_categorical]

# Only apply get_dummies to selected cols (others remain unchanged)
df_encoded = pd.get_dummies(df, columns=cat_cols_to_encode, drop_first=True)

print("Step 5: After one-hot encoding:")
print(df_encoded.head())

Step 5: After one-hot encoding:
   satisfaction  Gender  Age   Type of Travel  Flight Distance  Seat comfort  \
0             1  female   65  personal travel              265             0   
1             1    male   47  personal travel             2464             0   
2             1  female   15  personal travel             2138             0   
3             1  female   60  personal travel              623             0   
4             1  female   70  personal travel              354             0   

   Departure/Arrival time convenient  Food and drink  Gate location  \
0                                  0               0              2   
1                                  0               0              3   
2                                  0               0              3   
3                                  0               0              3   
4                                  0               0              3   

   Inflight wifi service  ...  Leg room service  Baggage han

In [80]:
# Step 6: Standardize numeric columns (adds new columns with '_std' suffix). Original columns stay unchanged

from sklearn.preprocessing import StandardScaler

# 1. Select all numeric columns to standardize
numeric_cols = df_encoded.select_dtypes(include='number').columns

# 2. Standardize - add new columns with '_std' suffix
scaler = StandardScaler()
std_values = scaler.fit_transform(df_encoded[numeric_cols])

# Add new columns; original columns remain unchanged
for i, col in enumerate(numeric_cols):
    df_encoded[col + '_std'] = std_values[:, i]

# Display to confirm original and standardized columns
print(df_encoded[[*numeric_cols, *(col+'_std' for col in numeric_cols)]].head())

   satisfaction  Age  Flight Distance  Seat comfort  \
0             1   65              265             0   
1             1   47             2464             0   
2             1   15             2138             0   
3             1   60              623             0   
4             1   70              354             0   

   Departure/Arrival time convenient  Food and drink  Gate location  \
0                                  0               0              2   
1                                  0               0              3   
2                                  0               0              3   
3                                  0               0              3   
4                                  0               0              3   

   Inflight wifi service  Inflight entertainment  Online support  ...  \
0                      2                       4               2  ...   
1                      0                       2               2  ...   
2                      

In [84]:
# Step 7: Look for Outliers

# Remove rows with any outliers, then drop *_outlier columns
if outlier_cols:
    df_encoded = df_encoded[~df_encoded[outlier_cols].any(axis=1)].reset_index(drop=True)
    
    # Drop *_outlier columns from the DataFrame
    df_encoded = df_encoded.drop(columns=outlier_cols)

# Even if no outliers found, it's safe to drop columns if present
else:
    # Find and drop any leftover *_outlier columns just in case
    outlier_cols_all = [col for col in df_encoded.columns if col.endswith('_outlier')]
    if outlier_cols_all:
        df_encoded = df_encoded.drop(columns=outlier_cols_all)

if outlier_cols:
    outliers_found = df_encoded[df_encoded[outlier_cols].any(axis=1)]
    if not outliers_found.empty:
        print("Outliers detected:")
        print(outliers_found)
    else:
        print("No outliers detected.")
else:
    print("No outlier columns found.")


No outlier columns found.


In [85]:
# Now save the changes to the dataset

df_encoded.to_csv("Invistico_Airline_cleaned.csv", index=False)
print('Dataset saved.')

Dataset saved.
